In [1]:
!pip install autogluon

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 20.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is still looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longe

In [2]:
import os
import joblib
import warnings
os.environ["OMP_NUM_THREADS"] = "1"
import numpy as np
import pandas as pd
from sklearn.svm import SVR
import matplotlib.pyplot as plt
from google.colab import files
from statsmodels.tsa.arima.model import ARIMA
from autogluon.tabular import TabularPredictor
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [3]:
# Open a file upload dialog
uploaded = files.upload()

Saving Cleaned Dataset.csv to Cleaned Dataset.csv


In [4]:
dataset = pd.read_csv("Cleaned Dataset.csv")

In [5]:
# Define the target
target = 'Log_BEV Percentage (Total Number Of Registrations)'

In [6]:
# Load the PCA tools
tools = joblib.load("pca_processors.pkl")
scaler = tools['scaler']
pca = tools['pca_model']
features = tools['feature_names']

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.3.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.3.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [7]:
# Data splitting into Pre and Post COVID
pre_covid_df = dataset[dataset['Year'] < 2020].copy()
post_covid_df = dataset[dataset['Year'] >= 2020].copy()

Please refer the jupyter notebook - **3.1.Feature Selection_BEV Percentage (Primary Target)** to understand the imputation

In [8]:
# Check which columns have NaNs and how many
print("Missing Values Per Column (Pre-COVID)")
print(pre_covid_df[features].isna().sum())

print("\nMissing Values Per Column (Post-COVID)")
print(post_covid_df[features].isna().sum())

Missing Values Per Column (Pre-COVID)
Log_Recharging Points             0
Real Range                        0
Log_Available                     0
YJ_Purchase price (EUR)           0
AC Recharging Speed (km/h)        0
Battery Capacity                  0
Log_DC Recharging Speed (km/h)    0
EG                                0
Log_GDP                           0
Log_CPI                           8
dtype: int64

Missing Values Per Column (Post-COVID)
Log_Recharging Points             0
Real Range                        0
Log_Available                     0
YJ_Purchase price (EUR)           0
AC Recharging Speed (km/h)        0
Battery Capacity                  0
Log_DC Recharging Speed (km/h)    0
EG                                0
Log_GDP                           0
Log_CPI                           1
dtype: int64


In [9]:
# Checking which rows are empty
print("Missing Log_CPI in Pre-COVID")
print(pre_covid_df[pre_covid_df['Log_CPI'].isna()].reset_index()[['Country', 'Year', 'Log_CPI']])
print("\n")
print("Missing Log_CPI in Post-COVID")
print(post_covid_df[post_covid_df['Log_CPI'].isna()].reset_index()[['Country', 'Year', 'Log_CPI']])

Missing Log_CPI in Pre-COVID
    Country  Year  Log_CPI
0  Bulgaria  2014      NaN
1   Croatia  2016      NaN
2    Cyprus  2014      NaN
3    Cyprus  2015      NaN
4    Cyprus  2016      NaN
5    Greece  2014      NaN
6    Greece  2015      NaN
7   Romania  2016      NaN


Missing Log_CPI in Post-COVID
  Country  Year  Log_CPI
0  Greece  2020      NaN


In [10]:
# Imputation for Log_CPI NaNs by filling the spaces with the median

# Calculate the median Log_CPI for every country of Pre-COVID data
country_medians = pre_covid_df.groupby('Country')['Log_CPI'].median()

# Function to fill the gaps
def impute_by_country(df):
    df_clean = df.copy()

    # Loop through the data, if Log_CPI is missing, we look up that specific country's median from our 'country_medians' list.
    df_clean['Log_CPI'] = df_clean['Log_CPI'].fillna(df_clean['Country'].map(country_medians))

    return df_clean

# Apply the fix to both datasets
pre_covid_fixed = impute_by_country(pre_covid_df)
post_covid_fixed = impute_by_country(post_covid_df)

# Extract 10 features
X_train_imputed = pre_covid_fixed[features]
X_test_imputed = post_covid_fixed[features]

In [11]:
# Scale the features using the scaler loaded from the .pkl file
X_train_scaled = scaler.transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

In [12]:
# Transform the features into 3 Principal Components using the loaded PCA model
# and convert them back to a dataframe for easy handling
X_train_pca = pd.DataFrame(pca.transform(X_train_scaled), columns=['PC1', 'PC2', 'PC3'], index=pre_covid_fixed.index)
X_test_pca = pd.DataFrame(pca.transform(X_test_scaled), columns=['PC1', 'PC2', 'PC3'], index=post_covid_fixed.index)

In [13]:
# Define the target
y_train = pre_covid_fixed[target]
y_test = post_covid_fixed[target]

print(f"Train Shape (PCs): {X_train_pca.shape}")
print(f"Test Shape (PCs): {X_test_pca.shape}")

Train Shape (PCs): (243, 3)
Test Shape (PCs): (135, 3)


In [14]:
# Define the clusters based on the K-Means clusters + PCA
leaders_list = [
    'Austria', 'Belgium', 'Czech Republic', 'Denmark', 'Finland', 'France',
    'Germany', 'Hungary', 'Italy', 'Netherlands', 'Poland',
    'Romania', 'Spain', 'Sweden'
]

followers_list = [
    'Bulgaria', 'Croatia', 'Cyprus', 'Estonia', 'Greece', 'Ireland',
    'Latvia', 'Lithuania', 'Luxembourg', 'Malta', 'Portugal', 'Slovakia', 'Slovenia'
]

In [15]:
# Filter Training Data (Pre-COVID)
X_train_leaders = X_train_pca[pre_covid_fixed['Country'].isin(leaders_list)]
y_train_leaders = y_train[pre_covid_fixed['Country'].isin(leaders_list)]

X_train_followers = X_train_pca[pre_covid_fixed['Country'].isin(followers_list)]
y_train_followers = y_train[pre_covid_fixed['Country'].isin(followers_list)]

# Filter Testing Data (Post-COVID)
X_test_leaders = X_test_pca[post_covid_fixed['Country'].isin(leaders_list)]
y_test_leaders = y_test[post_covid_fixed['Country'].isin(leaders_list)]

X_test_followers = X_test_pca[post_covid_fixed['Country'].isin(followers_list)]
y_test_followers = y_test[post_covid_fixed['Country'].isin(followers_list)]

## AUTOGLUON MODEL

**1. Explanation:** [AutoGluon](https://aws.amazon.com/blogs/opensource/machine-learning-with-autogluon-an-open-source-automl-library/)

**2. Documentation:** [Link](https://auto.gluon.ai/stable/index.html)

**3. GitHub Repository:** [Link](https://github.com/autogluon/autogluon)

In [16]:
# Prepare data function for AutoGluon: AutoGluon needs all the PCs and the target in a single DataFrame as its designed to work on Pandas dataframe.
def prepare_ag_data(X_pca, y_target):
    df = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(X_pca.shape[1])])
    df['target'] = y_target.values
    return df

train_data_ag = prepare_ag_data(X_train_pca, y_train)
test_data_ag = prepare_ag_data(X_test_pca, y_test)

In [17]:
# Train AutoGluon and fit the model
predictor = TabularPredictor(
    label='target', # 'Log_BEV Percentage (Total Number Of Registrations)'
    problem_type='regression', # Prediction of continuous numbers
    eval_metric='mean_absolute_error' # 'mean_absolute_error' because R2 is unstable for single-year tests
).fit(
    train_data_ag, # Training data
    presets='best_quality', # Enables stacking and ensembling for higher accuracy
    time_limit=600          # 10-minute training limit. Will try to train as many model combinations as possible within this window.
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260318_152437"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Mon Feb  2 12:27:57 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       11.11 GB / 12.67 GB (87.7%)
Disk Space Avail:   74.73 GB / 107.72 GB (69.4%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable sta

(autoscaler +36s) Tip: use `ray status` to view detailed cluster status. To disable these messages, set RAY_SCHEDULER_EVENTS=0.
(autoscaler +36s) Warning: The following resource request cannot be scheduled right now: {'CPU': 1.0}. This is likely due to all cluster resources being claimed by actors. Consider creating fewer actors or adding more nodes to this Ray cluster.
(_ray_fit pid=3053) [1000]	valid_set's l1: 0.17516
(_ray_fit pid=3053) [2000]	valid_set's l1: 0.172193


(_dystack pid=2604) 	-0.1755	 = Validation score   (-mean_absolute_error)
(_dystack pid=2604) 	32.86s	 = Training   runtime
(_dystack pid=2604) 	0.03s	 = Validation runtime
(_dystack pid=2604) Fitting model: LightGBM_BAG_L1 ... Training model for up to 47.08s of the 91.10s of remaining time.
(_dystack pid=2604) 	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (2 workers, per: cpus=1, gpus=0, memory=0.16%)


(autoscaler +1m11s) Warning: The following resource request cannot be scheduled right now: {'CPU': 1.0}. This is likely due to all cluster resources being claimed by actors. Consider creating fewer actors or adding more nodes to this Ray cluster.


(_dystack pid=2604) 	-0.1723	 = Validation score   (-mean_absolute_error)
(_dystack pid=2604) 	35.19s	 = Training   runtime
(_dystack pid=2604) 	0.01s	 = Validation runtime
(_dystack pid=2604) Fitting model: RandomForestMSE_BAG_L1 ... Training model for up to 6.75s of the 50.76s of remaining time.
(_dystack pid=2604) 	Fitting 1 model on all data (use_child_oof=True) | Fitting with cpus=2, gpus=0
(_dystack pid=2604) 	-0.1784	 = Validation score   (-mean_absolute_error)
(_dystack pid=2604) 	2.63s	 = Training   runtime
(_dystack pid=2604) 	0.43s	 = Validation runtime
(_dystack pid=2604) Fitting model: CatBoost_BAG_L1 ... Training model for up to 3.64s of the 47.66s of remaining time.
(_dystack pid=2604) 	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (2 workers, per: cpus=1, gpus=0, memory=3.87%)


(autoscaler +1m46s) Warning: The following resource request cannot be scheduled right now: {'CPU': 1.0}. This is likely due to all cluster resources being claimed by actors. Consider creating fewer actors or adding more nodes to this Ray cluster.


(_dystack pid=2604) 	Time limit exceeded... Skipping CatBoost_BAG_L1.
(_dystack pid=2604) Fitting model: WeightedEnsemble_L2 ... Training model for up to 132.01s of the 38.77s of remaining time.
(_dystack pid=2604) 	Fitting 1 model on all data | Fitting with cpus=2, gpus=0, mem=0.0/9.3 GB
(_dystack pid=2604) 	Ensemble Weights: {'LightGBM_BAG_L1': 0.6, 'RandomForestMSE_BAG_L1': 0.4}
(_dystack pid=2604) 	-0.1671	 = Validation score   (-mean_absolute_error)
(_dystack pid=2604) 	0.05s	 = Training   runtime
(_dystack pid=2604) 	0.0s	 = Validation runtime
(_dystack pid=2604) Fitting 106 L2 models, fit_strategy="sequential" ...
(_dystack pid=2604) Fitting model: LightGBMXT_BAG_L2 ... Training model for up to 38.71s of the 38.68s of remaining time.
(_dystack pid=2604) 	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (2 workers, per: cpus=1, gpus=0, memory=0.17%)
(_dystack pid=2604) 	-0.1708	 = Validation score   (-mean_absolute_error)
(_dystack pid=2604) 	2

(autoscaler +2m31s) Warning: The following resource request cannot be scheduled right now: {'CPU': 1.0}. This is likely due to all cluster resources being claimed by actors. Consider creating fewer actors or adding more nodes to this Ray cluster.


(_ray_fit pid=4423) 	Ran out of time, early stopping on iteration 1. Best iteration is:
(_ray_fit pid=4423) 	[1]	valid_set's l1: 0.274908
(_dystack pid=2604) 	-0.1901	 = Validation score   (-mean_absolute_error)
(_dystack pid=2604) 	30.85s	 = Training   runtime
(_dystack pid=2604) 	0.03s	 = Validation runtime
(_dystack pid=2604) Fitting model: WeightedEnsemble_L3 ... Training model for up to 132.01s of the -34.76s of remaining time.
(_dystack pid=2604) 	Fitting 1 model on all data | Fitting with cpus=2, gpus=0, mem=0.0/9.4 GB
(_dystack pid=2604) 	Ensemble Weights: {'LightGBMXT_BAG_L2': 0.833, 'RandomForestMSE_BAG_L1': 0.167}
(_dystack pid=2604) 	-0.169	 = Validation score   (-mean_absolute_error)
(_dystack pid=2604) 	0.06s	 = Training   runtime
(_dystack pid=2604) 	0.0s	 = Validation runtime
(_dystack pid=2604) AutoGluon training complete, total runtime = 166.88s ... Best model: WeightedEnsemble_L2 | Estimated inference throughput: 396.7 rows/s (27 batch size)
(_dystack pid=2604) Tabul

In [18]:
# View the Model Leaderboard
# This shows you which models (XGBoost, Neural Nets, etc.) performed best within the time frame
lb = predictor.leaderboard(test_data_ag, silent=True)
print("\n--- Model Leaderboard ---")
print(lb)


--- Model Leaderboard ---
                     model  score_test  score_val          eval_metric  \
0   NeuralNetFastAI_BAG_L1   -0.702434  -0.169716  mean_absolute_error   
1   NeuralNetFastAI_BAG_L2   -0.724326  -0.167922  mean_absolute_error   
2   RandomForestMSE_BAG_L1   -0.744187  -0.182882  mean_absolute_error   
3   RandomForestMSE_BAG_L2   -0.750947  -0.189453  mean_absolute_error   
4           XGBoost_BAG_L1   -0.760239  -0.177360  mean_absolute_error   
5      WeightedEnsemble_L2   -0.767788  -0.163107  mean_absolute_error   
6     ExtraTreesMSE_BAG_L1   -0.826818  -0.181217  mean_absolute_error   
7      WeightedEnsemble_L3   -0.865760  -0.160758  mean_absolute_error   
8     ExtraTreesMSE_BAG_L2   -0.920273  -0.182326  mean_absolute_error   
9     LightGBMLarge_BAG_L1   -0.935600  -0.188627  mean_absolute_error   
10         CatBoost_BAG_L1   -1.038591  -0.169535  mean_absolute_error   
11         LightGBM_BAG_L1   -1.095582  -0.180581  mean_absolute_error   
12       Li

**Explantion of the results**

The autogluon has checked on 17 different models

In [19]:
# Generate Predictions for 2020-2024
# We drop the 'target' column from the test data to predict it
# # Ensures the model is forced to guess the result using only the input features (PC1, PC2, PC3) without "cheating" by looking at the answer.
preds = predictor.predict(test_data_ag.drop(columns=['target']))

In [20]:
# Back-transform by converting log values back to real BEV percentages
y_test_real = np.exp(test_data_ag['target'])
preds_real = np.exp(preds)

In [21]:
# Create unified results dataframe
all_results = pd.DataFrame({
    'Country': post_covid_fixed['Country'],
    'Year': post_covid_fixed['Year'],
    'Actual_BEV_%': y_test_real,
    'Predicted_BEV_%': preds_real
})

# Absolute error
all_results['Error_Abs'] = (all_results['Actual_BEV_%'] - all_results['Predicted_BEV_%']).abs()

# Loop year-wise
for year, year_df in all_results.groupby('Year'):

    print(f"====>YEAR: {int(year)} PREDICTION RESULTS")

    # Leaders
    print(f"\n**** LEADERS CLUSTER ({int(year)}) ****")
    leaders_df = (
        year_df[year_df['Country'].isin(leaders_list)]
        .sort_values('Actual_BEV_%', ascending=False)
    )
    print(leaders_df[['Country', 'Actual_BEV_%', 'Predicted_BEV_%', 'Error_Abs']]
          .to_string(index=False))

    # Followers
    print(f"\n**** FOLLOWERS CLUSTER ({int(year)}) ****")
    followers_df = (
        year_df[year_df['Country'].isin(followers_list)]
        .sort_values('Actual_BEV_%', ascending=False)
    )
    print(followers_df[['Country', 'Actual_BEV_%', 'Predicted_BEV_%', 'Error_Abs']]
          .to_string(index=False))


====>YEAR: 2020 PREDICTION RESULTS

**** LEADERS CLUSTER (2020) ****
       Country  Actual_BEV_%  Predicted_BEV_%  Error_Abs
   Netherlands         21.43         2.155964  19.274036
        Sweden         10.55         1.632204   8.917796
       Denmark          8.16         1.551772   6.608228
        France          7.72         1.121246   6.598754
       Germany          7.56         1.661258   5.898742
       Austria          7.41         1.625333   5.784667
       Finland          5.40         1.335769   4.064231
       Belgium          4.48         1.681227   2.798773
         Italy          3.35         1.496531   1.853469
       Hungary          3.32         1.678911   1.641089
       Romania          3.25         1.486625   1.763375
         Spain          3.14         1.369057   1.770943
Czech Republic          2.61         1.645646   0.964354
        Poland          1.86         2.160566   0.300566

**** FOLLOWERS CLUSTER (2020) ****
   Country  Actual_BEV_%  Predicted_BEV_

In [22]:
# Filter the results as per the clusters
all_results['Cluster'] = all_results['Country'].apply(
    lambda x: 'Leaders' if x in leaders_list else 'Followers'
)

# Create the Leaders and Followers dataFrame and sort in alphabetical order for easier visualisation
leaders_results_df = (
    all_results[all_results['Cluster'] == 'Leaders']
    .sort_values(by=['Country', 'Year'])
)

followers_results_df = (
    all_results[all_results['Cluster'] == 'Followers']
    .sort_values(by=['Country', 'Year'])
)

# Save the data frame in an excel file
leaders_results_df.to_excel("Trial1_AutoGluon_Leaders_Results.xlsx", index=False)
followers_results_df.to_excel("Trial1_AutoGluon_Followers_Results.xlsx", index=False)